# Whole Barangay Paoay · farmland-first crop map

Audit actual cloud-screened Sentinel-2 coverage for January–April and May–August in 2024–2026. Use an indicative whole-barangay boundary, with the original smaller study area kept in its existing workflow.

The land-cover step preserves Dynamic World as a diagnostic. In the 2024 run, its repeated-crop rule found zero supported farmland pixels. The selected mask uses the published 2024 SEA_Cropland10 layer on the same 10 m grid. Only its mapped cropland goes to the frozen classifier, and the imagery still needs all eight half-months. The crop map gives non-farmland, uncertain land cover, missing imagery, and low-confidence crop type their own display categories.


In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import json
from zoneinfo import ZoneInfo

from crop_pipeline.analysis_workflow import resolve_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline.paoay import whole_barangay as barangay
from crop_pipeline.preprocessing_workflow import paths

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
_, report_root, _ = paths(root)
output = report_root.parent / "paoay" / "whole_barangay"
settings = json.loads((root / "settings.local.json").read_text())
today = datetime.now(ZoneInfo("Asia/Manila")).date()

BOUNDARY_INPUT = root / "data/paoay/paoay_barangay_osm.geojson"
QUERY_EARTH_ENGINE = False  # True runs a fresh cloud-screened coverage audit.
DOWNLOAD_MISSING = False  # True downloads verified missing inputs, without cloud exports.
WINDOW_START = "2024-01-01"
USE_REGIONAL_CROPLAND = True  # The 2024 layer applies only to 2024 imagery.
ALLOW_EXPLORATORY_CALENDAR_SHIFT = True  # The frozen fixed model trained on May–August readings.

reference = barangay.boundary_reference(barangay.fetch_boundary(BOUNDARY_INPUT), output / "references")
earth_engine = None
if QUERY_EARTH_ENGINE or DOWNLOAD_MISSING:
    import ee
    ee.Initialize(project=settings["earth_engine_project"])
    earth_engine = ee

if QUERY_EARTH_ENGINE:
    audit = barangay.audit(reference, output, earth_engine, today=today)
elif (output / "window_audit.csv").exists():
    with (output / "window_audit.csv").open() as stream:
        audit = list(csv.DictReader(stream))
else:
    audit = []
if audit:
    show_table(audit, title="Whole-barangay coverage before the land-cover mask")


## Classify supported farmland

The default period is January–April 2024. Applying the fixed forest in those months is an exploratory application with a different calendar from training. Choose a May start and switch off the calendar-shift flag for recipe-matched inference. A period that fails the saved 80% coverage gate over supported farmland keeps its coverage and land-cover outputs, with crop labels withheld. Field records are needed to validate both broad cropland and crop identity.


In [ ]:
model_index = resolve_stage_pointer(root / "pipeline_outputs/training/current_models.json")
model = model_index.parent / "fixed_matched/model_manifest.json"
handoff = barangay.apply_period(
    reference, WINDOW_START, model, output, ee=earth_engine, download=DOWNLOAD_MISSING,
    permit_shifted_fixed=ALLOW_EXPLORATORY_CALENDAR_SHIFT,
    use_regional_cropland=USE_REGIONAL_CROPLAND, today=today,
)
print("Map bundle:", handoff)
record = json.loads(handoff.read_text())
print("Application mode:", record["identity"]["mode"])
print("Cropland basis:", record["identity"]["cropland_basis"])
with (handoff.parent / "areas.csv").open() as stream:
    show_table(list(csv.DictReader(stream)), title="Area by map status")
barangay.plot_maps(handoff)
